# Kurationspipeline

In [1]:
from google.cloud import bigquery
import pandas as pd
from typing import Union
import re
import json

In [141]:
ror_ids = [
    {
        'name': 'Carl von Ossietzky Universität Oldenburg',
        'ror_id': 'https://ror.org/033n9gh91', 
    },
    {
        'name': 'Hochschule für Musik, Theater und Medien Hannover',
        'ror_id': 'https://ror.org/00x67m532', 
    },
    {
        'name': 'Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth',
        'ror_id': 'https://ror.org/02vvvm705', 
    },
    {
        'name': 'Universität Osnabrück',
        'ror_id': 'https://ror.org/04qmmjx98', 
    },
    {
        'name': 'Ostfalia Hochschule für angewandte Wissenschaften',
        'ror_id': 'https://ror.org/01bk10867', 
    },
    {
        'name': 'Gottfried Wilhelm Leibniz Universität Hannover',
        'ror_id': 'https://ror.org/0304hq317', 
    },
    {
        'name': 'Hochschule Hannover',
        'ror_id': 'https://ror.org/03m2kj587', 
    },
    {
        'name': 'Medizinische Hochschule Hannover (MHH)',
        'ror_id': 'https://ror.org/00f2yqf98', 
    },
    {
        'name': 'Georg-August-Universität Göttingen',
        'ror_id': 'https://ror.org/01y9bpm73', 
    },
    {
        'name': 'Technische Universität Braunschweig',
        'ror_id': 'https://ror.org/010nsgg66', 
    },
    {
        'name': 'Hochschule Emden/Leer',
        'ror_id': 'https://ror.org/01bc76c69', 
    },
    {
        'name': 'Stiftung Universität Hildesheim',
        'ror_id': 'https://ror.org/02f9det96', 
    },
    {
        'name': 'Stiftung Tierärztliche Hochschule Hannover',
        'ror_id': 'https://ror.org/015qjqf64', 
    },
    {
        'name': 'Technische Universität Clausthal',
        'ror_id': 'https://ror.org/04qb8nc58', 
    },
    {
        'name': 'Hochschule Osnabrück',
        'ror_id': 'https://ror.org/059vymd37', 
    },
    {
        'name': 'Hochschule für Bildende Künste Braunschweig',
        'ror_id': 'https://ror.org/03aft2f80', 
    },
    {
        'name': 'Universität Vechta',
        'ror_id': 'https://ror.org/045y6d111', 
    },
    {
        'name': 'Leuphana Universität Lüneburg',
        'ror_id': 'https://ror.org/02w2y2t16', 
    },
    {
        'name': 'HAWK Hochschule für angewandte Wissenschaft und Kunst',
        'ror_id': 'https://ror.org/00f5q5839', 
    },
    {
        'name': 'Universitätsmedizin Göttingen',
        'ror_id': 'https://ror.org/021ft0n22', 
    },
]

In [2]:
client = bigquery.Client(project='subugoe-collaborative')

In [3]:
oal_inst_lower_saxony_raw = client.query(f"""
                                          SELECT DISTINCT
                                          CASE 
                                            WHEN oal.id IS NOT NULL THEN oal.id
                                            ELSE kb.id
                                          END AS id,
                                          kb.source AS kb_source, 
                                          oal.source AS oal_source,
                                          kb.inst_name AS kb_name,
                                          oal.inst_name AS oal_name,
                                          kb.ror_id,
                                          CASE 
                                            WHEN oal.publication_year IS NOT NULL THEN oal.publication_year
                                            ELSE kb.publication_year
                                          END AS publication_year,
                                          CASE 
                                            WHEN oal.raw_affiliation_string IS NOT NULL THEN oal.raw_affiliation_string
                                            ELSE address_full
                                          END AS raw_affiliation_string
                                        FROM (
                                          SELECT o.id, 
                                                 CASE 
                                                   WHEN kb_inst.ror = 'https://ror.org/021ft0n22' THEN 'Universitätsmedizin Göttingen'
                                                   WHEN kb_inst.ror = 'https://ror.org/02w2y2t16' THEN 'Leuphana Universität Lüneburg'
                                                   WHEN kb_inst.ror = 'https://ror.org/033n9gh91' THEN 'Carl von Ossietzky Universität Oldenburg'
                                                   WHEN kb_inst.ror = 'https://ror.org/02vvvm705' THEN 'Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth'
                                                   WHEN kb_inst.ror = 'https://ror.org/01bc76c69' THEN 'Hochschule Emden/Leer'
                                                   WHEN kb_inst.ror = 'https://ror.org/0304hq317' THEN 'Gottfried Wilhelm Leibniz Universität Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/00x67m532' THEN 'Hochschule für Musik, Theater und Medien Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/03m2kj587' THEN 'Hochschule Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/015qjqf64' THEN 'Stiftung Tierärztliche Hochschule Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/00f2yqf98' THEN 'Medizinische Hochschule Hannover (MHH)'
                                                   WHEN kb_inst.ror = 'https://ror.org/00f5q5839' THEN 'HAWK Hochschule für angewandte Wissenschaft und Kunst'
                                                   WHEN kb_inst.ror = 'https://ror.org/02f9det96' THEN 'Stiftung Universität Hildesheim'
                                                   WHEN kb_inst.ror = 'https://ror.org/01y9bpm73' THEN 'Georg-August-Universität Göttingen'
                                                   WHEN kb_inst.ror = 'https://ror.org/010nsgg66' THEN 'Technische Universität Braunschweig'
                                                   WHEN kb_inst.ror = 'https://ror.org/03aft2f80' THEN 'Hochschule für Bildende Künste Braunschweig'
                                                   WHEN kb_inst.ror = 'https://ror.org/01bk10867' THEN 'Ostfalia Hochschule für angewandte Wissenschaften'
                                                   WHEN kb_inst.ror = 'https://ror.org/04qb8nc58' THEN 'Technische Universität Clausthal'
                                                   WHEN kb_inst.ror = 'https://ror.org/04qmmjx98' THEN 'Universität Osnabrück'
                                                   WHEN kb_inst.ror = 'https://ror.org/059vymd37' THEN 'Hochschule Osnabrück'
                                                   WHEN kb_inst.ror = 'https://ror.org/045y6d111' THEN 'Universität Vechta'
                                                   -- Ergänzung von Suborganisationen (Oldenburg)
                                                   WHEN inst_id = 445 THEN 'Carl von Ossietzky Universität Oldenburg' -- Evangelisches Krankenhaus Oldenburg
                                                   WHEN inst_id = 6612 THEN 'Carl von Ossietzky Universität Oldenburg' -- UMO - Universitätsmedizin Oldenburg
                                                   WHEN inst_id = 316 THEN 'Carl von Ossietzky Universität Oldenburg' -- Klinikum Oldenburg gGmbH
                                                   WHEN inst_id = 315 THEN 'Carl von Ossietzky Universität Oldenburg' -- Pius-Hospital Oldenburg
                                                   WHEN inst_id = 690 THEN 'Carl von Ossietzky Universität Oldenburg' -- Oldenburger Institut für Informatik
                                                   WHEN inst_id = 5488 THEN 'Carl von Ossietzky Universität Oldenburg' -- Helmholtz-Institut für Funktionelle Marine Biodiversität an der Universität Oldenburg (HIFMB)
                                                   WHEN inst_id = 4564 THEN 'Carl von Ossietzky Universität Oldenburg' -- Institute for Science Networking Oldenburg GmbH
                                                   -- Ergänzung von Suborganisationen (MHH)
                                                   WHEN inst_id = 6191 THEN 'Medizinische Hochschule Hannover (MHH)' -- Zentrum für Individualisierte Infektionsmedizin
                                                   WHEN inst_id = 4177 THEN 'Medizinische Hochschule Hannover (MHH)' -- Centre for Structural Systems Biology
                                                   WHEN inst_id = 6117 THEN 'Medizinische Hochschule Hannover (MHH)' -- Zentrum für Experimentelle und Klinische Infektionsforschung
                                                   ELSE ''
                                                 END AS inst_name,
                                                 kb_inst.ror AS ror_id,
                                                 address_full, 
                                                 -- REGEXP_REPLACE(address_full, r'[.,\s+]', '') AS raw_affiliation_string_cleaned,
                                                 publication_year, 
                                                 'KB' AS source
                                          FROM `subugoe-collaborative.resources.kb_a_addr_inst_202603` AS inst
                                          JOIN `subugoe-collaborative.resources.add_institution_lookup_kb_suppl_202603` AS kb_inst
                                            ON inst.inst_id_top = kb_inst.inst_id
                                          JOIN `subugoe-collaborative.openalex_walden.works` AS o
                                              ON CONCAT('https://openalex.org/', inst.item_id) = o.id
                                          WHERE o.type IN ('article', 'review') 
                                              AND primary_location.source.type = 'journal'
                                              AND is_paratext=FALSE 
                                              AND is_retracted=FALSE 
                                              AND is_xpac=FALSE
                                              AND publication_year BETWEEN 2020 AND 2024
                                              AND (kb_inst.ror IN (
                                                  'https://ror.org/021ft0n22', -- Universitätsmedizin Göttingen
                                                  'https://ror.org/02w2y2t16', -- Leuphana Universität Lüneburg
                                                  'https://ror.org/033n9gh91', -- Carl von Ossietzky Universität Oldenburg
                                                  'https://ror.org/02vvvm705', -- Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth
                                                  'https://ror.org/01bc76c69', -- Hochschule Emden/Leer
                                                  'https://ror.org/0304hq317', -- Gottfried Wilhelm Leibniz Universität Hannover
                                                  'https://ror.org/00x67m532', -- Hochschule für Musik, Theater und Medien Hannover
                                                  'https://ror.org/03m2kj587', -- Hochschule Hannover
                                                  'https://ror.org/015qjqf64', -- Stiftung Tierärztliche Hochschule Hannover
                                                  'https://ror.org/00f2yqf98', -- Medizinische Hochschule Hannover (MHH)
                                                  'https://ror.org/00f5q5839', -- HAWK Hochschule für angewandte Wissenschaft und Kunst
                                                  'https://ror.org/02f9det96', -- Stiftung Universität Hildesheim
                                                  'https://ror.org/01y9bpm73', -- Georg-August-Universität Göttingen
                                                  'https://ror.org/010nsgg66', -- Technische Universität Braunschweig
                                                  'https://ror.org/03aft2f80', -- Hochschule für Bildende Künste Braunschweig
                                                  'https://ror.org/01bk10867', -- Ostfalia Hochschule für angewandte Wissenschaften
                                                  'https://ror.org/04qb8nc58', -- Technische Universität Clausthal
                                                  'https://ror.org/04qmmjx98', -- Universität Osnabrück
                                                  'https://ror.org/059vymd37', -- Hochschule Osnabrück
                                                  'https://ror.org/045y6d111' -- Universität Vechta
                                              ) OR kb_inst.inst_id IN (
                                                    -- Ergänzung von Suborganisationen (Oldenburg)
                                                    445, -- 'Evangelisches Krankenhaus Oldenburg'
                                                    6612, -- 'UMO - Universitätsmedizin Oldenburg'
                                                    316, -- 'Klinikum Oldenburg gGmbH'
                                                    315, -- 'Pius-Hospital Oldenburg'
                                                    5488, -- 'Helmholtz-Institut für Funktionelle Marine Biodiversität an der Universität Oldenburg (HIFMB)'
                                                    4564, -- Institute for Science Networking Oldenburg GmbH
                                                    690, -- Oldenburger Institut für Informatik
                                                    -- Ergänzungen von Suborganisation (MHH)
                                                    6191, -- Zentrum für Individualisierte Infektionsmedizin
                                                    4177, -- Centre for Structural Systems Biology
                                                    6117 -- Zentrum für Experimentelle und Klinische Infektionsforschung
                                                )
                                              )
                                        ) AS kb
                                        FULL OUTER JOIN (
                                            SELECT oal.id, 
                                               CASE 
                                                 WHEN inst.ror = 'https://ror.org/02w2y2t16' THEN 'Leuphana Universität Lüneburg'
                                                 WHEN inst.ror = 'https://ror.org/033n9gh91' THEN 'Carl von Ossietzky Universität Oldenburg'
                                                 WHEN inst.ror = 'https://ror.org/02vvvm705' THEN 'Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth'
                                                 WHEN inst.ror = 'https://ror.org/01bc76c69' THEN 'Hochschule Emden/Leer'
                                                 WHEN inst.ror = 'https://ror.org/0304hq317' THEN 'Gottfried Wilhelm Leibniz Universität Hannover'
                                                 WHEN inst.ror = 'https://ror.org/00x67m532' THEN 'Hochschule für Musik, Theater und Medien Hannover'
                                                 WHEN inst.ror = 'https://ror.org/03m2kj587' THEN 'Hochschule Hannover'
                                                 WHEN inst.ror = 'https://ror.org/015qjqf64' THEN 'Stiftung Tierärztliche Hochschule Hannover'
                                                 WHEN inst.ror = 'https://ror.org/00f2yqf98' THEN 'Medizinische Hochschule Hannover (MHH)'
                                                 WHEN inst.ror = 'https://ror.org/00f5q5839' THEN 'HAWK Hochschule für angewandte Wissenschaft und Kunst'
                                                 WHEN inst.ror = 'https://ror.org/02f9det96' THEN 'Stiftung Universität Hildesheim'
                                                 WHEN inst.ror = 'https://ror.org/01y9bpm73' THEN 'Georg-August-Universität Göttingen'
                                                 WHEN inst.ror = 'https://ror.org/010nsgg66' THEN 'Technische Universität Braunschweig'
                                                 WHEN inst.ror = 'https://ror.org/03aft2f80' THEN 'Hochschule für Bildende Künste Braunschweig'
                                                 WHEN inst.ror = 'https://ror.org/01bk10867' THEN 'Ostfalia Hochschule für angewandte Wissenschaften'
                                                 WHEN inst.ror = 'https://ror.org/04qb8nc58' THEN 'Technische Universität Clausthal'
                                                 WHEN inst.ror = 'https://ror.org/04qmmjx98' THEN 'Universität Osnabrück'
                                                 WHEN inst.ror = 'https://ror.org/059vymd37' THEN 'Hochschule Osnabrück'
                                                 WHEN inst.ror = 'https://ror.org/045y6d111' THEN 'Universität Vechta'
                                                  -- Ergänzung von Suborganisationen (GAU)
                                                 WHEN inst.ror = 'https://ror.org/021ft0n22' THEN 'Georg-August-Universität Göttingen' -- UMG in GAU integrieren für Vergleichbarkeit
                                                 WHEN inst.ror = 'https://ror.org/00cd95c65' THEN 'Georg-August-Universität Göttingen' -- Gesellschaft für wissenschaftliche Datenverarbeitung mbH Göttingen
                                                 WHEN inst.ror = 'https://ror.org/02f04tm31' THEN 'Georg-August-Universität Göttingen' -- Göttingen Campus Institut für Dynamic biologischer Netzwerke
                                                 WHEN inst.ror = 'https://ror.org/044sxzm68' THEN 'Georg-August-Universität Göttingen' -- Campus-Institut Data Science (CIDAS)
                                                 WHEN inst.ror = 'https://ror.org/05745n787' THEN 'Georg-August-Universität Göttingen' -- Niedersächsische Staats-und Universitätsbibliothek Göttingen
                                                 WHEN inst.ror = 'https://ror.org/05xy1nn52' THEN 'Georg-August-Universität Göttingen' -- Multiscale Bioimaging
                                                 WHEN inst.ror = 'https://ror.org/03vwt8p73' THEN 'Georg-August-Universität Göttingen' -- Else Kröner Fresenius Zentrum für Optogenetische Therapien
                                                 WHEN inst.ror = 'https://ror.org/029w5ya68' THEN 'Georg-August-Universität Göttingen' -- European Neuroscience Institute Göttingen
                                                 WHEN inst.ror = 'https://ror.org/031q2en94' THEN 'Georg-August-Universität Göttingen' -- Volkswirtschaftliches Institut für Mittelstand und Handwerk
                                                 --WHEN inst.ror = 'https://ror.org/040pxfk62' THEN 'Georg-August-Universität Göttingen' -- Soziologisches Forschungsinstitut Göttingen
                                                 --WHEN inst.ror = 'https://ror.org/05831r008' THEN 'Georg-August-Universität Göttingen' -- Institut für Zuckerrübenforschung
                                                 --WHEN inst.ror = 'https://ror.org/03hpxd290' THEN 'Georg-August-Universität Göttingen' -- Nordwestdeutsche Forstliche Versuchsanstalt
                                                 --WHEN inst.ror = 'https://ror.org/003g6b432' THEN 'Georg-August-Universität Göttingen' -- Bernstein Zentrum für Computational Neuroscience Göttingen
                                                 -- Ergänzung von Suborganisationen (LUH)
                                                 WHEN inst.ror = 'https://ror.org/039t4wk02' THEN 'Gottfried Wilhelm Leibniz Universität Hannover' -- Forschungszentrum L3S
                                                 WHEN inst.ror = 'https://ror.org/00w53fs94' THEN 'Gottfried Wilhelm Leibniz Universität Hannover' -- Forschungszentrum Küste (FZK)
                                                 -- Ergänzung von Suborganisationen (Oldenburg)
                                                 WHEN inst.ror = 'https://ror.org/025t8vx68' THEN 'Carl von Ossietzky Universität Oldenburg' -- Institut für Ökonomische Bildung
                                                 WHEN inst.ror = 'https://ror.org/0060pja03' THEN 'Carl von Ossietzky Universität Oldenburg' -- Institut für Chemie und Biologie des Meeres
                                                 WHEN inst.ror = 'https://ror.org/003sav189' THEN 'Carl von Ossietzky Universität Oldenburg' -- Oldenburger Institut für Informatik
                                                 WHEN inst.ror = 'https://ror.org/01t0n2c80' THEN 'Carl von Ossietzky Universität Oldenburg' -- Klinikum Oldenburg
                                                 WHEN inst.ror = 'https://ror.org/04830hf15' THEN 'Carl von Ossietzky Universität Oldenburg' -- Evangelisches Krankenhaus Oldenburg
                                                 WHEN inst.ror = 'https://ror.org/03avbdx23' THEN 'Carl von Ossietzky Universität Oldenburg' -- Pius Hospital Oldenburg
                                                 WHEN inst.ror = 'https://ror.org/00tea5y39' THEN 'Carl von Ossietzky Universität Oldenburg' -- Helmholtz-Institut für Funktionelle Marine Biodiversität
                                                 -- Ergänzungen von Suborganisation (MHH)
                                                 WHEN inst.ror = 'https://ror.org/04s99xz91' THEN 'Medizinische Hochschule Hannover (MHH)' -- Zentrum für Individualisierte Infektionsmedizin
                                                 WHEN inst.ror = 'https://ror.org/04fhwda97' THEN 'Medizinische Hochschule Hannover (MHH)' -- Centre for Structural Systems Biology
                                                 WHEN inst.ror = 'https://ror.org/04bya8j72' THEN 'Medizinische Hochschule Hannover (MHH)' -- Zentrum für Experimentelle und Klinische Infektionsforschung
                                                 -- Ergänzungen von Suborganisation (Technische Universität Clausthal)
                                                 WHEN inst.ror = 'https://ror.org/00q7z2571' THEN 'Technische Universität Clausthal' -- Forschungszentrum Energiespeichertechnologien
                                                 ELSE ''
                                               END AS inst_name,
                                               inst.ror AS ror_id,
                                               raw_affiliation_string,
                                               -- REGEXP_REPLACE(raw_affiliation_string, r'[.,\s+]', '') AS raw_affiliation_string_cleaned,
                                               publication_year, 
                                               'OAL' AS source
                                            FROM `subugoe-collaborative.openalex_walden.works` AS oal
                                            LEFT JOIN UNNEST(authorships) AS aut
                                            LEFT JOIN UNNEST(aut.affiliations) AS aff
                                            LEFT JOIN UNNEST(institution_ids) AS aff_inst_id 
                                            JOIN `subugoe-collaborative.openalex_walden.institutions` AS inst
                                                -- ON aff.institution_ids[SAFE_OFFSET(0)] = inst.id
                                                ON aff_inst_id = inst.id
                                            WHERE oal.type IN ('article', 'review') 
                                                AND primary_location.source.type = 'journal'
                                                AND is_paratext=FALSE 
                                                AND is_retracted=FALSE 
                                                AND is_xpac=FALSE
                                                AND publication_year BETWEEN 2020 AND 2024
                                                AND inst.ror IN (
                                                    'https://ror.org/021ft0n22', -- Universitätsmedizin Göttingen
                                                    'https://ror.org/02w2y2t16', -- Leuphana Universität Lüneburg
                                                    'https://ror.org/033n9gh91', -- Carl von Ossietzky Universität Oldenburg
                                                    'https://ror.org/02vvvm705', -- Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth
                                                    'https://ror.org/01bc76c69', -- Hochschule Emden/Leer
                                                    'https://ror.org/0304hq317', -- Gottfried Wilhelm Leibniz Universität Hannover
                                                    'https://ror.org/00x67m532', -- Hochschule für Musik, Theater und Medien Hannover
                                                    'https://ror.org/03m2kj587', -- Hochschule Hannover
                                                    'https://ror.org/015qjqf64', -- Stiftung Tierärztliche Hochschule Hannover
                                                    'https://ror.org/00f2yqf98', -- Medizinische Hochschule Hannover (MHH)
                                                    'https://ror.org/00f5q5839', -- HAWK Hochschule für angewandte Wissenschaft und Kunst
                                                    'https://ror.org/02f9det96', -- Stiftung Universität Hildesheim
                                                    'https://ror.org/01y9bpm73', -- Georg-August-Universität Göttingen
                                                    'https://ror.org/010nsgg66', -- Technische Universität Braunschweig
                                                    'https://ror.org/03aft2f80', -- Hochschule für Bildende Künste Braunschweig
                                                    'https://ror.org/01bk10867', -- Ostfalia Hochschule für angewandte Wissenschaften
                                                    'https://ror.org/04qb8nc58', -- Technische Universität Clausthal
                                                    'https://ror.org/04qmmjx98', -- Universität Osnabrück
                                                    'https://ror.org/059vymd37', -- Hochschule Osnabrück
                                                    'https://ror.org/045y6d111', -- Universität Vechta
                                                    -- Ergänzung von Suborganisationen (GAU)
                                                    'https://ror.org/00cd95c65', -- Gesellschaft für wissenschaftliche Datenverarbeitung mbH Göttingen
                                                    'https://ror.org/02f04tm31', -- Göttingen Campus Institut für Dynamic biologischer Netzwerke
                                                    'https://ror.org/044sxzm68', -- Campus-Institut Data Science (CIDAS)
                                                    'https://ror.org/05745n787', -- Niedersächsische Staats-und Universitätsbibliothek Göttingen
                                                    'https://ror.org/05xy1nn52', -- Multiscale Bioimaging
                                                    'https://ror.org/03vwt8p73', -- Else Kröner Fresenius Zentrum für Optogenetische Therapien
                                                    'https://ror.org/029w5ya68', -- European Neuroscience Institute Göttingen
                                                    'https://ror.org/031q2en94', -- Volkswirtschaftliches Institut für Mittelstand und Handwerk
                                                    --'https://ror.org/040pxfk62', -- Soziologisches Forschungsinstitut Göttingen
                                                    --'https://ror.org/05831r008', -- Institut für Zuckerrübenforschung
                                                    --'https://ror.org/03hpxd290', -- Nordwestdeutsche Forstliche Versuchsanstalt
                                                    --'https://ror.org/003g6b432', -- Bernstein Zentrum für Computational Neuroscience Göttingen
                                                    -- Ergänzung von Suborganisationen (LUH)
                                                    'https://ror.org/039t4wk02', -- Forschungszentrum L3S
                                                    'https://ror.org/00w53fs94', -- Forschungszentrum Küste (FZK)
                                                    -- Ergänzung von Suborganisationen (Oldenburg)
                                                    'https://ror.org/025t8vx68', -- Institut für Ökonomische Bildung
                                                    'https://ror.org/0060pja03', -- Institut für Chemie und Biologie des Meeres
                                                    'https://ror.org/003sav189', -- Oldenburger Institut für Informatik
                                                    'https://ror.org/01t0n2c80', -- Klinikum Oldenburg
                                                    'https://ror.org/04830hf15', -- Evangelisches Krankenhaus Oldenburg
                                                    'https://ror.org/03avbdx23', -- Pius Hospital Oldenburg
                                                    'https://ror.org/00tea5y39', -- Helmholtz-Institut für Funktionelle Marine Biodiversität
                                                    -- Ergänzungen von Suborganisation (MHH)
                                                    'https://ror.org/04s99xz91', -- Zentrum für Individualisierte Infektionsmedizin
                                                    'https://ror.org/04fhwda97', -- Centre for Structural Systems Biology
                                                    'https://ror.org/04bya8j72', -- Zentrum für Experimentelle und Klinische Infektionsforschung
                                                    -- Ergänzungen von Suborganisation (Technische Universität Clausthal)
                                                    'https://ror.org/00q7z2571' -- Forschungszentrum Energiespeichertechnologien
                                                )
                                            ) AS oal
                                        ON kb.id = oal.id
                                        -- AND LOWER(kb.raw_affiliation_string_cleaned) = LOWER(oal.raw_affiliation_string_cleaned)
                                        -- AND kb.ror_id = oal.ror_id
                                        AND kb.inst_name = oal.inst_name
                            """).to_dataframe()

In [5]:
#oal_inst_lower_saxony_raw.to_csv('../data/inst_list_full_with_aff_strings.csv', sep=',', index=False)

In [6]:
oal_inst_lower_saxony = pd.read_csv('../data/inst_list_full_with_aff_strings.csv')

In [7]:
oal_inst_lower_saxony.head()

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
0,https://openalex.org/W4210261570,KB,OAL,Ostfalia Hochschule für angewandte Wissenschaften,Ostfalia Hochschule für angewandte Wissenschaften,https://ror.org/01bk10867,2022,Professorin für Wirtschaftspsychologie an dem ...
1,https://openalex.org/W4321241847,KB,OAL,Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth,Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth,https://ror.org/02vvvm705,2023,"Institute of Hearing Technology and Audiology,..."
2,https://openalex.org/W4390049664,KB,OAL,Universität Vechta,Universität Vechta,https://ror.org/045y6d111,2023,University of Vechta
3,https://openalex.org/W3131049324,KB,OAL,Universität Vechta,Universität Vechta,https://ror.org/045y6d111,2021,"University of Vechta, Neuer Markt 32, 49377 Ve..."
4,https://openalex.org/W4400689112,KB,OAL,HAWK Hochschule für angewandte Wissenschaft un...,HAWK Hochschule für angewandte Wissenschaft un...,https://ror.org/00f5q5839,2024,"HAWK University of Applied Sciences and Arts, ..."


In [8]:
def make_set(list_of_names: list): 
    return set(name for name in list_of_names if pd.notna(name))

In [9]:
kb_list = oal_inst_lower_saxony.groupby(['id', 'raw_affiliation_string', 'publication_year'])['kb_name'].apply(make_set).reset_index()
oal_list = oal_inst_lower_saxony.groupby(['id', 'raw_affiliation_string', 'publication_year'])['oal_name'].apply(make_set).reset_index()
inst_list = pd.merge(kb_list, oal_list, on=['id', 'raw_affiliation_string', 'publication_year'], how='outer')

In [10]:
inst_list['kb_name'] = inst_list['kb_name'].fillna('').apply(make_set)
inst_list['oal_name'] = inst_list['oal_name'].fillna('').apply(make_set)

inst_list['in_oal_missing'] = list(inst_list['kb_name'] - inst_list['oal_name'])
inst_list['in_kb_missing'] = list(inst_list['oal_name'] - inst_list['kb_name'])

inst_list['kb_count'] = inst_list.kb_name.str.len()
inst_list['oal_count'] = inst_list.oal_name.str.len()

In [11]:
inst_list.head()

,id,raw_affiliation_string,publication_year,kb_name,oal_name,in_oal_missing,in_kb_missing,kb_count,oal_count
0,https://openalex.org/W112007689,Götting KG,2024,{},{Georg-August-Universität Göttingen},{},{Georg-August-Universität Göttingen},0,1
1,https://openalex.org/W112007689,Institut für Transport- und Automatisierungste...,2024,{Gottfried Wilhelm Leibniz Universität Hannover},{Gottfried Wilhelm Leibniz Universität Hannover},{},{},1,1
2,https://openalex.org/W1483587807,"Centre Georg Simmel, Recherches franco-alleman...",2021,{Leuphana Universität Lüneburg},{Leuphana Universität Lüneburg},{},{},1,1
3,https://openalex.org/W1483587807,Leuphana University Lueneburg,2021,{Leuphana Universität Lüneburg},{Leuphana Universität Lüneburg},{},{},1,1
4,https://openalex.org/W1500095539,"University and Polytechnic of Lüneburg, German...",2024,{},{Leuphana Universität Lüneburg},{},{Leuphana Universität Lüneburg},0,1


In [12]:
df = inst_list[['id', 'raw_affiliation_string', 'publication_year', 'in_oal_missing', 'in_kb_missing']].copy()

In [14]:
df = df.assign(in_oal_missing=df['in_oal_missing']).explode('in_oal_missing').reset_index(drop=True)
df = df.assign(in_kb_missing=df['in_kb_missing']).explode('in_kb_missing').reset_index(drop=True)

In [15]:
df

,id,raw_affiliation_string,publication_year,in_oal_missing,in_kb_missing
0,https://openalex.org/W112007689,Götting KG,2024,NaN,Georg-August-Universität Göttingen
1,https://openalex.org/W112007689,Institut für Transport- und Automatisierungste...,2024,NaN,NaN
2,https://openalex.org/W1483587807,"Centre Georg Simmel, Recherches franco-alleman...",2021,NaN,NaN
3,https://openalex.org/W1483587807,Leuphana University Lueneburg,2021,NaN,NaN
4,https://openalex.org/W1500095539,"University and Polytechnic of Lüneburg, German...",2024,NaN,Leuphana Universität Lüneburg
...,...,...,...,...,...
112597,https://openalex.org/W7155016106,"University of Vechta, Faculty II, Geography & ...",2024,NaN,Universität Vechta
112598,https://openalex.org/W7160830967,"University of Goettingen, Germany",2024,NaN,Georg-August-Universität Göttingen
112599,https://openalex.org/W7161541681,former docente at the Institute of Music at th...,2024,NaN,Carl von Ossietzky Universität Oldenburg
112600,https://openalex.org/W7163685763,"Fachbereich BGG, Jade Hochschule Oldenburg, Ge...",2024,NaN,Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth


In [16]:
df.dropna(subset=['in_oal_missing', 'in_kb_missing'], how='all', inplace=True)

In [17]:
df = df[~df.publication_year.isnull()]
df['publication_year'] = df['publication_year'].astype(int)

In [18]:
df.head()

,id,raw_affiliation_string,publication_year,in_oal_missing,in_kb_missing
0,https://openalex.org/W112007689,Götting KG,2024,NaN,Georg-August-Universität Göttingen
4,https://openalex.org/W1500095539,"University and Polytechnic of Lüneburg, German...",2024,NaN,Leuphana Universität Lüneburg
11,https://openalex.org/W173619620,Universität Lüneburg Zentrum für Angewandte Ge...,2024,Leuphana Universität Lüneburg,NaN
27,https://openalex.org/W2016255857,Götting KG,2024,NaN,Georg-August-Universität Göttingen
36,https://openalex.org/W2233989344,ocupa lugar,2020,NaN,Leuphana Universität Lüneburg


In [19]:
df[(~df.in_kb_missing.isnull()) & (df.in_kb_missing != 'nan')]

,id,raw_affiliation_string,publication_year,in_oal_missing,in_kb_missing
0,https://openalex.org/W112007689,Götting KG,2024,NaN,Georg-August-Universität Göttingen
4,https://openalex.org/W1500095539,"University and Polytechnic of Lüneburg, German...",2024,NaN,Leuphana Universität Lüneburg
27,https://openalex.org/W2016255857,Götting KG,2024,NaN,Georg-August-Universität Göttingen
36,https://openalex.org/W2233989344,ocupa lugar,2020,NaN,Leuphana Universität Lüneburg
38,https://openalex.org/W2261575631,"Oldenburg, 1967, 284",2021,NaN,Carl von Ossietzky Universität Oldenburg
...,...,...,...,...,...
112596,https://openalex.org/W7155016106,"University of Vechta, Faculty II, Biology & Ve...",2024,NaN,Universität Vechta
112597,https://openalex.org/W7155016106,"University of Vechta, Faculty II, Geography & ...",2024,NaN,Universität Vechta
112598,https://openalex.org/W7160830967,"University of Goettingen, Germany",2024,NaN,Georg-August-Universität Göttingen
112599,https://openalex.org/W7161541681,former docente at the Institute of Music at th...,2024,NaN,Carl von Ossietzky Universität Oldenburg


In [20]:
true_df = df.copy()
true_df['in_kb_missing'] = true_df['in_kb_missing'].replace('nan', '')
true_df['in_oal_missing'] = true_df['in_oal_missing'].replace('nan', '')

In [21]:
true_df.id.count()

7502

In [22]:
# Universitätsmedizin Göttingen
# https://ror.org/021ft0n22

with open('institution_mapping/mapping_tables_fak/gau.json', 'r') as file:
    GAU_MAPPING = json.load(file)

umg = GAU_MAPPING.get('Universitätsmedizin Göttingen (UMG)')

In [24]:
def map_umg(address: str) -> Union[str, None]:
    pattern = re.compile(r'|'.join(umg), re.IGNORECASE)
    res = bool(pattern.search(address))
    if res:
        return  'Universitätsmedizin Göttingen'
    else:
        return 'Georg-August-Universität Göttingen'

In [25]:
true_df.loc[true_df['in_oal_missing'] == 'Georg-August-Universität Göttingen', 'in_oal_missing'] = \
           true_df.loc[true_df['in_oal_missing'] == 'Georg-August-Universität Göttingen']['raw_affiliation_string'].apply(map_umg)

true_df.loc[true_df['in_kb_missing'] == 'Georg-August-Universität Göttingen', 'in_kb_missing'] = \
           true_df.loc[true_df['in_kb_missing'] == 'Georg-August-Universität Göttingen']['raw_affiliation_string'].apply(map_umg)

In [26]:
true_df

,id,raw_affiliation_string,publication_year,in_oal_missing,in_kb_missing
0,https://openalex.org/W112007689,Götting KG,2024,NaN,Georg-August-Universität Göttingen
4,https://openalex.org/W1500095539,"University and Polytechnic of Lüneburg, German...",2024,NaN,Leuphana Universität Lüneburg
11,https://openalex.org/W173619620,Universität Lüneburg Zentrum für Angewandte Ge...,2024,Leuphana Universität Lüneburg,NaN
27,https://openalex.org/W2016255857,Götting KG,2024,NaN,Georg-August-Universität Göttingen
36,https://openalex.org/W2233989344,ocupa lugar,2020,NaN,Leuphana Universität Lüneburg
...,...,...,...,...,...
112596,https://openalex.org/W7155016106,"University of Vechta, Faculty II, Biology & Ve...",2024,NaN,Universität Vechta
112597,https://openalex.org/W7155016106,"University of Vechta, Faculty II, Geography & ...",2024,NaN,Universität Vechta
112598,https://openalex.org/W7160830967,"University of Goettingen, Germany",2024,NaN,Georg-August-Universität Göttingen
112599,https://openalex.org/W7161541681,former docente at the Institute of Music at th...,2024,NaN,Carl von Ossietzky Universität Oldenburg


In [70]:
candidates = true_df.groupby(['raw_affiliation_string', 'in_oal_missing', 'in_kb_missing'], dropna=False)['id'].count().reset_index()

In [71]:
candidates

,raw_affiliation_string,in_oal_missing,in_kb_missing,id
0,\nClaudine Auger est journaliste.\n,NaN,Technische Universität Clausthal,1
1,\nDocteure en neurosciences et journaliste à O...,NaN,Carl von Ossietzky Universität Oldenburg,1
2,"\nDocteure en neurosciences et journaliste, à ...",NaN,Carl von Ossietzky Universität Oldenburg,1
3,\nRWI – Leibniz Institute for Economic Researc...,Georg-August-Universität Göttingen,NaN,1
4,\nUniversité d’Hildesheim\n,NaN,Stiftung Universität Hildesheim,1
...,...,...,...,...
5971,wissenschaftlicher Mitarbeiter am Lehrstuhl fü...,Georg-August-Universität Göttingen,NaN,1
5972,¶Department of Systems Immunology and Braunsch...,Technische Universität Braunschweig,NaN,1
5973,"Šumarski fakultet i Ekologija šuma, Sveučilišt...",NaN,Georg-August-Universität Göttingen,1
5974,Дніпровський національний університет імені Ол...,NaN,Universität Vechta,1


In [72]:
candidates.loc[candidates['in_oal_missing'].isnull(), 'to remove'] = candidates.loc[candidates['in_oal_missing'].isnull()].in_kb_missing
candidates.loc[candidates['in_kb_missing'].isnull(), 'to add'] = candidates.loc[candidates['in_kb_missing'].isnull()].in_oal_missing
candidates.loc[(~candidates['in_kb_missing'].isnull()) & (~candidates['in_oal_missing'].isnull()), 'to change'] = \
    candidates.loc[(~candidates['in_kb_missing'].isnull()) & (~candidates['in_oal_missing'].isnull())].in_oal_missing 

In [73]:
candidates

,raw_affiliation_string,in_oal_missing,in_kb_missing,id,to remove,to add,to change
0,\nClaudine Auger est journaliste.\n,NaN,Technische Universität Clausthal,1,Technische Universität Clausthal,NaN,NaN
1,\nDocteure en neurosciences et journaliste à O...,NaN,Carl von Ossietzky Universität Oldenburg,1,Carl von Ossietzky Universität Oldenburg,NaN,NaN
2,"\nDocteure en neurosciences et journaliste, à ...",NaN,Carl von Ossietzky Universität Oldenburg,1,Carl von Ossietzky Universität Oldenburg,NaN,NaN
3,\nRWI – Leibniz Institute for Economic Researc...,Georg-August-Universität Göttingen,NaN,1,NaN,Georg-August-Universität Göttingen,NaN
4,\nUniversité d’Hildesheim\n,NaN,Stiftung Universität Hildesheim,1,Stiftung Universität Hildesheim,NaN,NaN
...,...,...,...,...,...,...,...
5971,wissenschaftlicher Mitarbeiter am Lehrstuhl fü...,Georg-August-Universität Göttingen,NaN,1,NaN,Georg-August-Universität Göttingen,NaN
5972,¶Department of Systems Immunology and Braunsch...,Technische Universität Braunschweig,NaN,1,NaN,Technische Universität Braunschweig,NaN
5973,"Šumarski fakultet i Ekologija šuma, Sveučilišt...",NaN,Georg-August-Universität Göttingen,1,Georg-August-Universität Göttingen,NaN,NaN
5974,Дніпровський національний університет імені Ол...,NaN,Universität Vechta,1,Universität Vechta,NaN,NaN


In [74]:
candidates[['raw_affiliation_string', 'to add']].dropna()

,raw_affiliation_string,to add
3,\nRWI – Leibniz Institute for Economic Researc...,Georg-August-Universität Göttingen
7,(Leibniz Institute for Baltic Sea Research; Ca...,Carl von Ossietzky Universität Oldenburg
26,. Department of Anesthesiology and Intensive C...,Universitätsmedizin Göttingen
28,. Institute of Forest and Nature Conservation ...,Georg-August-Universität Göttingen
29,". Plastic and Reconstructive Surgery, Klinikum...",Universitätsmedizin Göttingen
...,...,...
5959,"reprint requests to Thomas Lenarz, M.D., Ph.D....",Gottfried Wilhelm Leibniz Universität Hannover
5969,vormals Akademischer Rat a. Z. am Lehrstuhl fü...,Gottfried Wilhelm Leibniz Universität Hannover
5970,wissenschaftlicher Mitarbeiter am Lehrstuhl fü...,Georg-August-Universität Göttingen
5971,wissenschaftlicher Mitarbeiter am Lehrstuhl fü...,Georg-August-Universität Göttingen


In [75]:
candidates[['raw_affiliation_string', 'to remove']].dropna()

,raw_affiliation_string,to remove
0,\nClaudine Auger est journaliste.\n,Technische Universität Clausthal
1,\nDocteure en neurosciences et journaliste à O...,Carl von Ossietzky Universität Oldenburg
2,"\nDocteure en neurosciences et journaliste, à ...",Carl von Ossietzky Universität Oldenburg
4,\nUniversité d’Hildesheim\n,Stiftung Universität Hildesheim
5,"""Hearing4all"", Hannover Medi-cal School, Hanov...",Medizinische Hochschule Hannover (MHH)
...,...,...
5967,"versity Hospital Oldenburg, Oldenburg, Germany;",Carl von Ossietzky Universität Oldenburg
5968,von der Hochschule Hannover kandidierte nicht ...,Hochschule Hannover
5973,"Šumarski fakultet i Ekologija šuma, Sveučilišt...",Georg-August-Universität Göttingen
5974,Дніпровський національний університет імені Ол...,Universität Vechta


In [88]:
change_candidates = candidates[['raw_affiliation_string', 'in_kb_missing', 'to change']].dropna()

In [89]:
change_candidates.columns = ['raw_affiliation_string', 'to remove', 'to add']

In [90]:
change_candidates

,raw_affiliation_string,to remove,to add
113,"7Department of Media, Information, and Design,...",Stiftung Tierärztliche Hochschule Hannover,Hochschule Hannover
237,Araththy Logeswaran hat im Fachbereich Erziehu...,Hochschule Osnabrück,Universität Osnabrück
274,Beispiel des deutschen und ungarischen Strafre...,Georg-August-Universität Göttingen,Universität Osnabrück
396,"Center for Biomolecular Drug Research (BMWZ), ...",Medizinische Hochschule Hannover (MHH),Gottfried Wilhelm Leibniz Universität Hannover
501,"Chair of Banking and Finance, Osnabrück Univer...",Hochschule Osnabrück,Universität Osnabrück
...,...,...,...
5644,University of Applied Sciences and Arts Hildes...,Stiftung Universität Hildesheim,HAWK Hochschule für angewandte Wissenschaft un...
5654,"University of Applied Sciences and Arts, Facul...",Georg-August-Universität Göttingen,HAWK Hochschule für angewandte Wissenschaft un...
5659,"University of Applied Sciences and Arts, Hilde...",Stiftung Universität Hildesheim,HAWK Hochschule für angewandte Wissenschaft un...
5666,"University of Applied Sciences, Plant Nutritio...",Universität Osnabrück,Hochschule Osnabrück


In [142]:
to_remove = pd.concat([candidates[['raw_affiliation_string', 'to remove']].dropna(), 
                       change_candidates[['raw_affiliation_string', 'to remove']]], ignore_index=True)

to_add = pd.concat([candidates[['raw_affiliation_string', 'to add']].dropna(), 
                    change_candidates[['raw_affiliation_string', 'to add']]], ignore_index=True)

In [143]:
def add_ror(university_string: str) -> str:
    
    for dicts in ror_ids:
        if dicts['name'] == university_string:  
            return dicts['ror_id']

In [144]:
to_remove['ror_id'] = to_remove['to remove'].apply(add_ror)
to_add['ror_id'] = to_add['to add'].apply(add_ror)

In [145]:
to_remove

,raw_affiliation_string,to remove,ror_id
0,\nClaudine Auger est journaliste.\n,Technische Universität Clausthal,https://ror.org/04qb8nc58
1,\nDocteure en neurosciences et journaliste à O...,Carl von Ossietzky Universität Oldenburg,https://ror.org/033n9gh91
2,"\nDocteure en neurosciences et journaliste, à ...",Carl von Ossietzky Universität Oldenburg,https://ror.org/033n9gh91
3,\nUniversité d’Hildesheim\n,Stiftung Universität Hildesheim,https://ror.org/02f9det96
4,"""Hearing4all"", Hannover Medi-cal School, Hanov...",Medizinische Hochschule Hannover (MHH),https://ror.org/00f2yqf98
...,...,...,...
2601,University of Applied Sciences and Arts Hildes...,Stiftung Universität Hildesheim,https://ror.org/02f9det96
2602,"University of Applied Sciences and Arts, Facul...",Georg-August-Universität Göttingen,https://ror.org/01y9bpm73
2603,"University of Applied Sciences and Arts, Hilde...",Stiftung Universität Hildesheim,https://ror.org/02f9det96
2604,"University of Applied Sciences, Plant Nutritio...",Universität Osnabrück,https://ror.org/04qmmjx98


In [146]:
to_add

,raw_affiliation_string,to add,ror_id
0,\nRWI – Leibniz Institute for Economic Researc...,Georg-August-Universität Göttingen,https://ror.org/01y9bpm73
1,(Leibniz Institute for Baltic Sea Research; Ca...,Carl von Ossietzky Universität Oldenburg,https://ror.org/033n9gh91
2,. Department of Anesthesiology and Intensive C...,Universitätsmedizin Göttingen,https://ror.org/021ft0n22
3,. Institute of Forest and Nature Conservation ...,Georg-August-Universität Göttingen,https://ror.org/01y9bpm73
4,". Plastic and Reconstructive Surgery, Klinikum...",Universitätsmedizin Göttingen,https://ror.org/021ft0n22
...,...,...,...
3457,University of Applied Sciences and Arts Hildes...,HAWK Hochschule für angewandte Wissenschaft un...,https://ror.org/00f5q5839
3458,"University of Applied Sciences and Arts, Facul...",HAWK Hochschule für angewandte Wissenschaft un...,https://ror.org/00f5q5839
3459,"University of Applied Sciences and Arts, Hilde...",HAWK Hochschule für angewandte Wissenschaft un...,https://ror.org/00f5q5839
3460,"University of Applied Sciences, Plant Nutritio...",Hochschule Osnabrück,https://ror.org/059vymd37


In [147]:
to_remove.to_csv('../data/curation_raw/to_remove.csv', index=False)
to_add.to_csv('../data/curation_raw/to_add.csv', index=False)